# Parte 4: Integración Final
## Riesgo de Contaminación por Nitratos | La Rioja, 2015 - 2025

*Nota:* La estrategia de únion es `merge(on='ipa', how='left')`, las variables espaciales y NDVI son atributos estáticos/anuales del punto. Cada observación hereda los atributos de su punto IPA. Sin `sjoin_nearest`.

In [1]:
import re
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from pathlib import Path
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [2]:
BASE_DIR = Path(r"C:\Users\mcangulo\OneDrive - FEDERACION DE EMPRESAS DE LA RIOJA\Escritorio\dataset_larioja")
OUT_DIR  = BASE_DIR / '9_dataset_final'
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUTA_P1    = OUT_DIR / 'dataset_nitratos_clima_larioja_final.csv'
RUTA_P2    = OUT_DIR / 'puntos_rnit_larioja_variables_espaciales_limpio.gpkg'
RUTA_P3    = OUT_DIR / 'puntos_rnit_con_ndvi.gpkg'
DIR_NDVI   = BASE_DIR / '8_ndvi_sentinel2'

print('P1 existe:', RUTA_P1.exists())
print('P2 existe:', RUTA_P2.exists())
print('P3 existe:', RUTA_P3.exists())

P1 existe: True
P2 existe: True
P3 existe: True


In [3]:
def normalizar_columnas(df):
    df = df.copy()
    df.columns = (df.columns.astype(str).str.strip().str.lower()
                  .str.replace('á','a').str.replace('é','e').str.replace('í','i')
                  .str.replace('ó','o').str.replace('ú','u').str.replace('ñ','n')
                  .str.replace(' ','_').str.replace('-','_'))
    return df

def limpiar_ipa(s):
    return s.astype(str).str.strip().str.replace(r'\.0$','',regex=True)

def resumen_nulos(df, top=20):
    n = df.isna().sum().sort_values(ascending=False).head(top)
    n = n[n>0]
    return pd.DataFrame({'columna':n.index,'n_nulos':n.values,'pct':(100*n.values/len(df)).round(1)})

print('Utilidades cargadas')

Utilidades cargadas


### 1. Cargar P1: observaciones nitratos + clima

Granularidad: se observa que solo tiene 1 fila por IPA × fecha

In [4]:
df_p1 = pd.read_csv(RUTA_P1)
df_p1 = normalizar_columnas(df_p1)
df_p1['fecha'] = pd.to_datetime(df_p1['fecha'], errors='coerce')
df_p1['ipa']   = limpiar_ipa(df_p1['ipa'])

print(f'P1: {len(df_p1):,} filas × {df_p1.shape[1]} columnas')
print(f'Puntos únicos: {df_p1.ipa.nunique()}')
print(f'Rango: {df_p1.fecha.min().date()} → {df_p1.fecha.max().date()}')
display(df_p1[['ipa','fecha','no3_mgl','clase','prec','precip_7d']].head(5))

P1: 1,173 filas × 38 columnas
Puntos únicos: 101
Rango: 2015-03-10 → 2025-12-17


,ipa,fecha,no3_mgl,clase,prec,precip_7d
0,211020002,2015-03-10,184.0,afectada,0.0,9.8
1,211020002,2015-09-08,155.0,afectada,0.0,37.4
2,211030008,2015-09-23,3.7,normal,0.0,1.0
3,211020002,2015-12-01,117.0,afectada,0.0,21.4
4,211020002,2016-03-08,117.0,afectada,1.4,19.0


### 2. Cargar P2 + P3: variables espaciales y NDVI

Granularidad: punto 1 fila por IPA

In [5]:
# Preferir P3 (incluye NDVI) sobre P2
ruta_esp = RUTA_P3 if RUTA_P3.exists() else RUTA_P2
print('Cargando:', ruta_esp.name)

gdf_p2 = gpd.read_file(ruta_esp)
gdf_p2 = normalizar_columnas(gdf_p2)

for col_id in ['ipa','id_punto','idptoinven']:
    if col_id in gdf_p2.columns:
        if col_id != 'ipa': gdf_p2 = gdf_p2.rename(columns={col_id:'ipa'})
        break
gdf_p2['ipa'] = limpiar_ipa(gdf_p2['ipa'])

# Limpiar columnas duplicadas lat/lon si las hay
for suf in ['_x','_y','_p2']:
    cols_dup = [c for c in gdf_p2.columns if c.endswith(suf) and c.replace(suf,'') in gdf_p2.columns]
    if cols_dup:
        gdf_p2 = gdf_p2.drop(columns=cols_dup)

# Correcciones SIGPAC
if 'sigpac_uso' in gdf_p2.columns:
    MT_DESC = {'MT':'Monte bajo / Matorral'}
    if 'sigpac_uso_nombre' in gdf_p2.columns:
        mask_mt = (gdf_p2['sigpac_uso']=='MT') & (gdf_p2['sigpac_uso_nombre']=='No clasificado')
        gdf_p2.loc[mask_mt,'sigpac_uso_nombre'] = 'Monte bajo / Matorral'
if 'sigpac_coef_regadio' in gdf_p2.columns:
    gdf_p2['sigpac_coef_regadio'] = gdf_p2['sigpac_coef_regadio'].fillna(0.0)

print(f'P2/P3: {len(gdf_p2)} puntos × {gdf_p2.shape[1]} columnas')
cats = {'SoilGrids':[c for c in gdf_p2.columns if c.startswith('soil_')],
        'CORINE':   [c for c in gdf_p2.columns if 'clc' in c],
        'SIGPAC':   [c for c in gdf_p2.columns if c.startswith('sigpac_')],
        'Zona':     [c for c in gdf_p2.columns if 'zona_vuln' in c],
        'Altitud':  [c for c in gdf_p2.columns if 'altitud_dem' in c],
        'NDVI':     [c for c in gdf_p2.columns if 'ndvi' in c],}
for cat, cols in cats.items():
    if cols: print(f'  {cat:<12}: {len(cols)} vars')

Cargando: puntos_rnit_con_ndvi.gpkg
P2/P3: 104 puntos × 68 columnas
  SoilGrids   : 24 vars
  CORINE      : 7 vars
  SIGPAC      : 6 vars
  Zona        : 1 vars
  Altitud     : 1 vars
  NDVI        : 15 vars


### 3. Verificar cobertura IPA

In [6]:
ipas_p1 = set(df_p1['ipa'].unique())
ipas_p2 = set(gdf_p2['ipa'].unique())
comunes = ipas_p1 & ipas_p2

print(f'IPAs en P1: {len(ipas_p1)}')
print(f'IPAs en P2: {len(ipas_p2)}')
print(f'IPAs comunes: {len(comunes)}')
print(f'P1 sin match: {len(ipas_p1-ipas_p2)}')

pct_sin = 100 * df_p1['ipa'].isin(ipas_p1-ipas_p2).mean()
if pct_sin > 15:
    print(f'AVISO: {pct_sin:.1f}% de observaciones sin datos espaciales')
    print('Revisar que P2 fue generada desde el mismo output de P1')
else:
    print(f'OK: {pct_sin:.1f}% sin match espacial — aceptable')

IPAs en P1: 101
IPAs en P2: 104
IPAs comunes: 101
P1 sin match: 0
OK: 0.0% sin match espacial — aceptable


### 4. Integración: merge por IPA

In [7]:
EXCLUIR_P2 = {'geometry','fecha','anio','mes','no3_mgl','afectada',
              'en_riesgo','clase','prec','tmed'}

df_p2_tabla = pd.DataFrame(gdf_p2.drop(columns=['geometry'],errors='ignore')).copy()
df_p2_tabla = df_p2_tabla.drop_duplicates('ipa', keep='first')
cols_p2 = ['ipa'] + [c for c in df_p2_tabla.columns if c!='ipa' and c not in EXCLUIR_P2]
df_p2_tabla = df_p2_tabla[cols_p2]

df_final = df_p1.merge(df_p2_tabla, on='ipa', how='left', suffixes=('','_esp'))

assert len(df_final)==len(df_p1), f'ERROR: filas cambiaron {len(df_p1)}→{len(df_final)}'

print(f'Merge OK: {len(df_final):,} filas (idéntico a P1)')
print(f'Columnas totales: {df_final.shape[1]}')
print(f'Variables añadidas: {len(df_final.columns)-len(df_p1.columns)}')

Merge OK: 1,173 filas (idéntico a P1)
Columnas totales: 104
Variables añadidas: 66


### 5. Asignar NDVI del año de la observación

Es la única variable verdaderamente dinámica de toda la integración: a cada fila (observación) se le asigna el valor de NDVI correspondiente exactamente a su año de muestreo, no el promedio del período ni el de otro año. Las columnas anuales en bruto (`ndvi_2015`…`ndvi_2025`) no se usarán como features del modelo, precisamente para evitar que el modelo "vea" información de años posteriores a la observación que está prediciendo.

In [8]:
# Asignar NDVI del año de cada observación (única variable dinámica)
ndvi_cols_anio = sorted([c for c in df_final.columns if re.fullmatch(r'ndvi_20\d{2}',c)])

if ndvi_cols_anio and 'anio' in df_final.columns:
    anio_num = pd.to_numeric(df_final['anio'], errors='coerce').astype('Int64')
    ndvi_obs = []
    for anio, row_ndvi in zip(anio_num, df_final[ndvi_cols_anio].values):
        col = f'ndvi_{int(anio)}' if pd.notna(anio) else None
        if col and col in ndvi_cols_anio:
            ndvi_obs.append(row_ndvi[ndvi_cols_anio.index(col)])
        else:
            ndvi_obs.append(np.nan)
    df_final['ndvi_anio_observacion'] = ndvi_obs
    cob = 100*pd.Series(ndvi_obs).notna().mean()
    print(f'NDVI del año de la obs: {cob:.1f}% con dato')
else:
    df_final['ndvi_anio_observacion'] = np.nan
    print('Sin columnas NDVI disponibles')

NDVI del año de la obs: 100.0% con dato


### 6. Diagnóstico del dataset final

Se revisa la cobertura de cada bloque de variables (clima, suelo, uso del suelo, SIGPAC, zona vulnerable, altitud, NDVI) y se identifican las columnas con valores nulos restantes, como verificación final antes de definir target y features.

In [9]:
print('=== DIAGNÓSTICO DATASET FINAL ===')
print(f'Filas:    {len(df_final):,}')
print(f'Columnas: {df_final.shape[1]}')
print(f'Puntos:   {df_final.ipa.nunique()}')
print()

grupos = {
    'Clima':      [c for c in df_final.columns if any(c.startswith(p) for p in ['prec','tmed','temp_','anomalia','evento'])],
    'Suelo':      [c for c in df_final.columns if c.startswith('soil_')],
    'CLC/Uso':    [c for c in df_final.columns if any(p in c for p in ['clc_','codigo_clc','uso_'])],
    'SIGPAC':     [c for c in df_final.columns if c.startswith('sigpac_')],
    'Zona':       [c for c in df_final.columns if 'zona_vuln' in c],
    'Altitud':    [c for c in df_final.columns if 'altitud_dem' in c],
    'NDVI':       [c for c in df_final.columns if 'ndvi' in c],
}
print('Cobertura por categoría (% obs con dato):')
for grupo, cols in grupos.items():
    if not cols: print(f'  {grupo:<12}: sin variables'); continue
    pct = 100*df_final[cols].notna().any(axis=1).mean()
    print(f'  {grupo:<12}: {len(cols):>3} vars | {pct:.0f}%')

print()
display(resumen_nulos(df_final, top=15))

=== DIAGNÓSTICO DATASET FINAL ===
Filas:    1,173
Columnas: 105
Puntos:   101

Cobertura por categoría (% obs con dato):
  Clima       :   9 vars | 100%
  Suelo       :  24 vars | 100%
  CLC/Uso     :   2 vars | 100%
  SIGPAC      :   6 vars | 100%
  Zona        :   1 vars | 100%
  Altitud     :   1 vars | 100%
  NDVI        :  16 vars | 100%



,columna,n_nulos,pct
0,localidad,464,39.6
1,cota,426,36.3
2,soil_phh2o_15_30cm,82,7.0
3,soil_phh2o_0_5cm,82,7.0
4,soil_phh2o_5_15cm,82,7.0
5,clc2018_nombre,12,1.0
6,clc2012_nombre,12,1.0
7,precip_30d,1,0.1
8,temp_media_30d,1,0.1


### 7. Separación de target y features para modelado

Se define explícitamente la variable objetivo (`clase`) y se excluyen del conjunto de features tanto las variables de fuga (`no3_mgl`, `afectada`, `en_riesgo`) como las columnas NDVI anuales en bruto, dejando solo `ndvi_anio_observacion` y los resúmenes del período (`ndvi_media_periodo`, `ndvi_cambio_periodo`, etc.).

In [10]:
TARGET = 'clase'

EXCLUIR_MODELO = {
    'ipa','fecha','nombre_punto','municipio','provincia','masa_agua',
    'codigo_masa_agua','red','indicativo_estacion','nombre_estacion',
    'no3_mgl','afectada','en_riesgo','clase',          # target y derivados
    'geometry',
    *[c for c in df_final.columns if re.fullmatch(r'ndvi_20\d{2}',c)],  # raw NDVI anuales
}

features = [c for c in df_final.columns if c not in EXCLUIR_MODELO]

X = df_final[features]
y = df_final[TARGET]

print(f'Target: {TARGET}')
print(f'Features: {len(features)}')
cats = {'Clima':[f for f in features if any(f.startswith(p) for p in ['prec','tmed','temp','anomalia','evento'])],
        'Suelo':[f for f in features if f.startswith('soil_') or f in ['textura_usda','agua_disponible_cm3cm3']],
        'Uso':  [f for f in features if any(p in f for p in ['clc_','sigpac_','agricola'])],
        'Terr': [f for f in features if f in ['altitud_dem_m','zona_vulnerable_nitratos','dist_estacion_m']],
        'NDVI': [f for f in features if 'ndvi' in f],
        'Esp':  [f for f in features if f in ['x_utm30','y_utm30']],}
for cat,cols in cats.items():
    if cols: print(f'  {cat}: {cols}')
print(f'\nX: {X.shape} | y: {y.shape}')
print('Distribución target:')
print(y.value_counts(normalize=True).mul(100).round(1))

Target: clase
Features: 82
  Clima: ['prec', 'tmed', 'precip_7d', 'precip_15d', 'precip_30d', 'temp_media_7d', 'temp_media_30d', 'anomalia_temp', 'evento_extremo_lluvia']
  Suelo: ['soil_cec_0_5cm', 'soil_cec_15_30cm', 'soil_cec_5_15cm', 'soil_clay_0_5cm', 'soil_clay_15_30cm', 'soil_clay_5_15cm', 'soil_phh2o_0_5cm', 'soil_phh2o_15_30cm', 'soil_phh2o_5_15cm', 'soil_sand_0_5cm', 'soil_sand_15_30cm', 'soil_sand_5_15cm', 'soil_silt_0_5cm', 'soil_silt_15_30cm', 'soil_silt_5_15cm', 'soil_soc_0_5cm', 'soil_soc_15_30cm', 'soil_soc_5_15cm', 'soil_wv0033_0_5cm', 'soil_wv0033_15_30cm', 'soil_wv0033_5_15cm', 'soil_wv1500_0_5cm', 'soil_wv1500_15_30cm', 'soil_wv1500_5_15cm']
  Uso: ['clc2012_agricola', 'clc2018_agricola', 'cambio_clc_2012_2018', 'sigpac_uso', 'sigpac_uso_nombre', 'sigpac_superficie_m2', 'sigpac_pendiente_pct', 'sigpac_coef_regadio', 'sigpac_es_agricola', 'dist_sigpac_m']
  Terr: ['dist_estacion_m', 'zona_vulnerable_nitratos', 'altitud_dem_m']
  NDVI: ['ndvi_media_periodo', 'ndvi_min

### 8. Guardado

In [ ]:
ruta_csv   = OUT_DIR / 'dataset_final_integrado_larioja_2015_2025.csv'
ruta_excel = OUT_DIR / 'dataset_final_integrado_larioja_2015_2025.xlsx'
ruta_gpkg  = OUT_DIR / 'dataset_final_integrado_larioja_2015_2025.gpkg'

df_final.to_csv(ruta_csv, index=False, encoding='utf-8-sig', sep=';')
df_final.to_excel(ruta_excel, index=False)

for col_x, col_y, crs in [('lon','lat','EPSG:4326'),('x_utm30','y_utm30','EPSG:25830')]:
    if {col_x,col_y}.issubset(df_final.columns):
        gpd.GeoDataFrame(df_final,
            geometry=gpd.points_from_xy(df_final[col_x],df_final[col_y]),crs=crs
        ).to_file(ruta_gpkg, driver='GPKG', layer='dataset_final')
        break

print('Dataset final guardado')
print('CSV:  ', ruta_csv)
print('GPKG: ', ruta_gpkg)
print(f'Filas: {df_final.shape[0]:,} | Columnas: {df_final.shape[1]}')
print()